# 06 — Walk-Forward Backtest (God Mode v3)Simulate the full trading strategy using walk-forward test predictions.Supports both XGBoost Stage 2 and direct TCN+GRU (bypass) modes.**Features:**- Next-bar-open entry (realistic fills)- Multi-leg cost model (maker/taker/slippage/funding/impact)- Fractional Kelly position sizing with leverage- SL protection (cooldown, price distance, consecutive cap)- Portfolio risk management (daily/weekly loss limits)- Adaptive TP/SL volatility scaling

In [ ]:
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml tqdm pyarrow matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2', '__init__.py')):
    raise FileNotFoundError('scalp2 package not found after clone!')

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scalp2.config import load_config
from scalp2.execution.trade_manager import TradeManager, TradeState, TradeStatus
from scalp2.execution.risk_manager import RiskManager
from scalp2.utils.metrics import sharpe_ratio, sortino_ratio, max_drawdown, win_rate, profit_factor

config = load_config(f'{REPO_DIR}/config.yaml')
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'

print(f'bypass_xgboost = {getattr(config.model, "bypass_xgboost", False)}')

In [ ]:
# Load labeled dataset and walk-forward predictions
import pickle

df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')

# Choose predictions based on bypass mode
bypass = getattr(config.model, 'bypass_xgboost', False)
if bypass:
    pkl_file = 'wf_predictions_stage1.pkl'
    print('GOD MODE: Using raw TCN+GRU (Stage 1) predictions — XGBoost bypassed')
else:
    pkl_file = 'wf_predictions.pkl'
    print('Standard mode: Using XGBoost (Stage 2) predictions')

pkl_path = f'{DATA_DIR}/{pkl_file}'
if not os.path.exists(pkl_path):
    raise FileNotFoundError(f'{pkl_file} not found! Run NB 05.2 (bypass) or NB 05 (standard) first.')

with open(pkl_path, 'rb') as f:
    wf_predictions = pickle.load(f)

print(f'Loaded {len(df):,} bars, {len(wf_predictions)} walk-forward folds')
print(f'Predictions source: {pkl_file}')

In [ ]:
# ============================================================
#  WALK-FORWARD BACKTEST ENGINE
#  Clean implementation with all cost layers and risk management
# ============================================================
from tqdm import tqdm

# Suppress log spam
logging.getLogger('scalp2.execution.trade_manager').setLevel(logging.WARNING)
logging.getLogger('scalp2.execution.risk_manager').setLevel(logging.WARNING)

seq_len = config.model.seq_len
exec_cfg = config.execution
trade_mgmt_cfg = config.execution.trade_management
label_cfg = config.labeling
order_cfg = config.execution.order_execution

trade_mgr = TradeManager(trade_mgmt_cfg, label_cfg.max_holding_bars)
risk_mgr = RiskManager(config=exec_cfg)

# --- Transaction costs ---
MAKER_FEE_BPS = order_cfg.maker_fee_bps
TAKER_FEE_BPS = order_cfg.taker_fee_bps
SLIPPAGE_BPS = order_cfg.slippage_bps
LEVERAGE = exec_cfg.position_sizing.leverage
MAINT_MARGIN = exec_cfg.position_sizing.maintenance_margin

# --- Variable slippage ---
slip_cfg = getattr(exec_cfg, 'slippage_model', None)
USE_VAR_SLIPPAGE = slip_cfg is not None and slip_cfg.enabled
MEDIAN_ATR = df['atr_14'].median() if USE_VAR_SLIPPAGE and 'atr_14' in df.columns else 1.0

def get_slippage_bps(atr_val):
    if not USE_VAR_SLIPPAGE:
        return SLIPPAGE_BPS
    return slip_cfg.base_bps + slip_cfg.volatility_scale * (atr_val / (MEDIAN_ATR + 1e-10))

# --- Funding rate ---
funding_cfg = getattr(exec_cfg, 'funding_rate', None)
USE_FUNDING = funding_cfg is not None and funding_cfg.enabled

def count_funding_intervals(entry_time, exit_time):
    if not USE_FUNDING:
        return 0
    funding_times = pd.date_range(
        entry_time.normalize(),
        exit_time.normalize() + pd.Timedelta(days=1),
        freq='8h'
    )
    return int(((funding_times > entry_time) & (funding_times <= exit_time)).sum())

# --- Market impact ---
impact_cfg = getattr(exec_cfg, 'market_impact', None)
USE_MARKET_IMPACT = impact_cfg is not None and impact_cfg.enabled

def market_impact_frac(position_size, price):
    if not USE_MARKET_IMPACT:
        return 0.0
    notional = position_size * price * LEVERAGE
    return impact_cfg.base_impact_bps * (notional / impact_cfg.reference_notional_usd) / 10_000

# Cost functions
def entry_cost_frac(slip_bps):
    return (MAKER_FEE_BPS + slip_bps) / 10_000

def exit_cost_frac(status, slip_bps):
    if status in (TradeStatus.CLOSED_SL, TradeStatus.CLOSED_TIME,
                  TradeStatus.CLOSED_REGIME, 'CLOSED_FOLD_END'):
        return (TAKER_FEE_BPS + slip_bps) / 10_000
    return (MAKER_FEE_BPS + slip_bps) / 10_000

FLAT_RT_COST = 2 * (SLIPPAGE_BPS + MAKER_FEE_BPS) / 10_000

# --- Kelly sizing ---
partial_pct = trade_mgmt_cfg.partial_tp_1_pct
partial_atr = trade_mgmt_cfg.partial_tp_1_atr
full_tp_atr = trade_mgmt_cfg.full_tp_atr
effective_tp = partial_pct * partial_atr + (1 - partial_pct) * full_tp_atr
kelly_b = effective_tp / label_cfg.sl_multiplier
kelly_fraction = exec_cfg.position_sizing.kelly_fraction
kelly_max = exec_cfg.position_sizing.max_fraction

# --- Filters ---
CONF_THRESHOLD = exec_cfg.confidence_threshold
MIN_ADX = exec_cfg.min_adx
MIN_ATR_PCTILE = exec_cfg.min_atr_percentile
CHOPPY_THRESHOLD = config.regime.choppy_threshold

# ATR percentile
if 'atr_14' in df.columns:
    df['atr_pctile'] = df['atr_14'].rolling(96, min_periods=10).rank(pct=True).fillna(1.0)
else:
    df['atr_pctile'] = 1.0

print(f'Mode: {"BYPASS (Stage 1)" if bypass else "XGBoost (Stage 2)"}')
print(f'TX: maker={MAKER_FEE_BPS}bps taker={TAKER_FEE_BPS}bps slip={SLIPPAGE_BPS}bps flat_RT={FLAT_RT_COST*10000:.0f}bps')
print(f'Leverage: {LEVERAGE}x | Kelly: fraction={kelly_fraction}, max={kelly_max}')
print(f'Filters: conf>={CONF_THRESHOLD}, adx>={MIN_ADX}, atr_pctile>={MIN_ATR_PCTILE}, choppy<{CHOPPY_THRESHOLD}')

# ═══════════════════════════════════════════════════════════════
#  BACKTEST LOOP
# ═══════════════════════════════════════════════════════════════
all_trades = []
equity_curve = [0.0]
bar_equity_curve = [0.0]
cumulative_pnl = 0.0
skip_reasons = {}
liquidated = False

def _close_trade(active, position_size, entry_bar, bar, row, fold_idx,
                 status_override=None, gross_override=None):
    global cumulative_pnl
    gross = gross_override if gross_override is not None else active.pnl
    status_val = status_override if status_override else active.status.value

    slip_bps = get_slippage_bps(active.atr_at_entry)
    cost = entry_cost_frac(slip_bps)
    if active.partial_fills:
        cost += exit_cost_frac(TradeStatus.CLOSED_TP, slip_bps) * partial_pct
        exit_status = status_override or active.status
        cost += exit_cost_frac(exit_status, slip_bps) * (1 - partial_pct)
    else:
        exit_status = status_override or active.status
        cost += exit_cost_frac(exit_status, slip_bps)

    impact = market_impact_frac(position_size, active.entry_price)
    cost += impact

    entry_ts = df.index[entry_bar]
    exit_ts = row.name
    n_funding = count_funding_intervals(entry_ts, exit_ts)
    funding = n_funding * (funding_cfg.fixed_rate_pct / 100.0) if USE_FUNDING else 0.0

    unit_net = (gross - cost - funding) * position_size
    leveraged_net = unit_net * LEVERAGE
    cumulative_pnl += leveraged_net

    all_trades.append(dict(
        fold=fold_idx, direction=active.direction,
        entry_price=active.entry_price, bars_held=active.bars_held,
        status=status_val,
        gross_pnl=gross * position_size * LEVERAGE,
        unit_pnl=unit_net, net_pnl=leveraged_net,
        cost=cost * position_size * LEVERAGE,
        funding_cost=funding * position_size * LEVERAGE,
        position_size=position_size,
        margin_utilization=position_size * LEVERAGE,
        slippage_bps=slip_bps, impact_bps=impact * 10_000,
        n_exits=1 + len(active.partial_fills),
        entry_bar=entry_bar, exit_bar=bar,
        timestamp=row.name,
    ))
    equity_curve.append(cumulative_pnl)

for fold_data in tqdm(wf_predictions, desc='Backtesting folds'):
    if liquidated:
        break

    fold_idx = fold_data['fold_idx']
    test_start = fold_data['test_start']
    test_end = fold_data['test_end']
    preds = fold_data['test_probabilities']
    n_preds = len(preds)
    pred_offset = test_start + seq_len

    regime_probs = fold_data.get('regime_probs', None)
    has_regime = regime_probs is not None

    active = None
    entry_bar = None
    pending_signal = None
    daily_count = 0
    prev_date = None

    for i in range(n_preds):
        bar = pred_offset + i
        if bar >= len(df):
            break

        row = df.iloc[bar]
        cur_date = row.name.date() if hasattr(row.name, 'date') else None
        if cur_date != prev_date:
            daily_count = 0
            prev_date = cur_date

        # Advance bar counter for SL protection
        trade_mgr.advance_bar()

        # ── Manage active trade ──
        if active is not None:
            is_choppy = False
            if has_regime and i < len(regime_probs):
                is_choppy = regime_probs[i, 2] > CHOPPY_THRESHOLD

            active = trade_mgr.update(active, row['high'], row['low'], row['close'], is_choppy)
            if active.status not in (TradeStatus.OPEN, TradeStatus.PARTIAL_TP):
                # Record for SL protection
                trade_mgr.record_trade_result(
                    status=active.status, exit_price=row['close'],
                    direction=active.direction, atr=active.atr_at_entry,
                )
                # Record in risk manager
                _trade_pnl_pct = active.pnl * position_size * LEVERAGE * 100
                risk_mgr.record_trade(
                    timestamp=row.name if hasattr(row.name, 'date') else pd.Timestamp(row.name),
                    pnl_pct=_trade_pnl_pct,
                )
                _close_trade(active, position_size, entry_bar, bar, row, fold_idx)
                if cumulative_pnl <= -1.0:
                    print(f'LIQUIDATED at fold {fold_idx}, bar {bar}')
                    liquidated = True
                    break
                active = None
            continue

        # ── Execute pending signal ──
        if pending_signal is not None:
            ps = pending_signal
            pending_signal = None
            entry_price = row['open']
            atr = ps['atr']
            direction = ps['direction']
            confidence = ps['confidence']

            if direction == "LONG":
                sl = entry_price - label_cfg.sl_multiplier * atr
            else:
                sl = entry_price + label_cfg.sl_multiplier * atr

            p = confidence
            q = 1 - p
            kelly = max((p * kelly_b - q) / kelly_b, 0)
            position_size = min(kelly * kelly_fraction, kelly_max)

            # Win streak reduction
            _size_mod = risk_mgr.get_position_size_modifier()
            if _size_mod < 1.0:
                position_size *= _size_mod

            if position_size < 1e-6:
                continue

            active = TradeState(
                direction=direction, entry_price=entry_price,
                current_stop_loss=sl, take_profit=0.0, atr_at_entry=atr,
            )
            entry_bar = bar
            daily_count += 1
            continue

        # ── Check for new signal ──
        p_arr = preds[i]
        cls = int(np.argmax(p_arr))
        if cls == 1:
            skip_reasons['hold'] = skip_reasons.get('hold', 0) + 1
            continue
        if max(p_arr[0], p_arr[2]) < CONF_THRESHOLD:
            skip_reasons['low_conf'] = skip_reasons.get('low_conf', 0) + 1
            continue
        if daily_count >= exec_cfg.max_trades_per_day:
            skip_reasons['daily_cap'] = skip_reasons.get('daily_cap', 0) + 1
            continue

        # Time-of-day filter
        tdf = getattr(exec_cfg, 'time_of_day_filter', None)
        if tdf and tdf.enabled:
            hr = row.name.hour if hasattr(row.name, 'hour') else pd.to_datetime(row.name).hour
            if hr in tdf.blocked_hours_utc:
                skip_reasons['time_blocked'] = skip_reasons.get('time_blocked', 0) + 1
                continue

        atr = row['atr_14'] if 'atr_14' in df.columns else 0.0
        if atr < 1e-10:
            skip_reasons['no_atr'] = skip_reasons.get('no_atr', 0) + 1
            continue

        # ADX filter
        adx_val = row['adx'] if 'adx' in df.columns else 999.0
        if adx_val < MIN_ADX:
            skip_reasons['low_adx'] = skip_reasons.get('low_adx', 0) + 1
            continue

        # ATR percentile filter
        atr_pct = row['atr_pctile'] if 'atr_pctile' in df.columns else 1.0
        if atr_pct < MIN_ATR_PCTILE:
            skip_reasons['low_volatility'] = skip_reasons.get('low_volatility', 0) + 1
            continue

        # Choppy regime filter
        if has_regime and i < len(regime_probs):
            if regime_probs[i, 2] > CHOPPY_THRESHOLD:
                skip_reasons['choppy'] = skip_reasons.get('choppy', 0) + 1
                continue

        # Check next bar exists
        next_bar = pred_offset + i + 1
        if next_bar >= len(df):
            skip_reasons['no_next_bar'] = skip_reasons.get('no_next_bar', 0) + 1
            continue

        direction = "LONG" if cls == 2 else "SHORT"

        # SL protection check
        can_enter, _skip = trade_mgr.can_enter_trade(
            direction=direction, entry_price=row['close'], current_atr=atr,
        )
        if not can_enter:
            skip_reasons[_skip] = skip_reasons.get(_skip, 0) + 1
            continue

        # Risk manager check
        _cur_ts = row.name if hasattr(row.name, 'date') else pd.Timestamp(row.name)
        _can_risk, _risk_reason = risk_mgr.can_trade(timestamp=_cur_ts)
        if not _can_risk:
            _short = _risk_reason.split(' ')[0]
            skip_reasons[_short] = skip_reasons.get(_short, 0) + 1
            continue

        pending_signal = {
            'direction': direction, 'atr': atr,
            'confidence': max(p_arr[0], p_arr[2]),
        }

        # M2M equity
        if active is not None:
            if active.direction == "LONG":
                unr = (row['close'] - active.entry_price) / active.entry_price
            else:
                unr = (active.entry_price - row['close']) / active.entry_price
            m_pnl = (active.pnl + unr * active.remaining_size) * position_size * LEVERAGE
            bar_equity_curve.append(cumulative_pnl + m_pnl)
        else:
            bar_equity_curve.append(cumulative_pnl)

    # Force-close at fold boundary
    if active is not None:
        last_row = df.iloc[min(test_end - 1, len(df) - 1)]
        if active.direction == "LONG":
            unr = (last_row['close'] - active.entry_price) / active.entry_price
        else:
            unr = (active.entry_price - last_row['close']) / active.entry_price
        gross = active.pnl + unr * active.remaining_size
        _close_trade(active, position_size, entry_bar,
                     min(test_end - 1, len(df) - 1), last_row, fold_idx,
                     status_override='CLOSED_FOLD_END', gross_override=gross)
        active = None
    pending_signal = None

trades_df = pd.DataFrame(all_trades)
print(f'\nTotal trades: {len(trades_df)}')
print(f'Cumulative net PnL ({LEVERAGE}x levered): {cumulative_pnl*100:.2f}%')
print(f'\nSkip reasons: {skip_reasons}')
if len(trades_df) > 0:
    print(f'Avg position size: {trades_df["position_size"].mean():.4f}')
    print(f'Avg margin util: {trades_df["margin_utilization"].mean()*100:.1f}%')
    if liquidated:
        print('*** ACCOUNT LIQUIDATED ***')

In [ ]:
# ============================================================
#  RESULTS SUMMARY + VISUALIZATION
# ============================================================
if len(trades_df) == 0:
    print("No trades generated!")
else:
    net = trades_df['net_pnl'].values
    gross = trades_df['gross_pnl'].values
    n = len(trades_df)

    wins = net[net > 0]
    losses = net[net < 0]
    wr = len(wins) / n
    avg_win = wins.mean() if len(wins) else 0
    avg_loss = losses.mean() if len(losses) else 0
    pf = abs(wins.sum() / losses.sum()) if len(losses) else float('inf')

    trades_df['date'] = pd.to_datetime(trades_df['timestamp']).dt.date
    daily_pnl = trades_df.groupby('date')['net_pnl'].sum()
    full_range = pd.date_range(daily_pnl.index.min(), daily_pnl.index.max(), freq='D')
    daily_pnl = daily_pnl.reindex(full_range, fill_value=0.0)

    daily_sharpe = daily_pnl.mean() / (daily_pnl.std() + 1e-10) * np.sqrt(365)
    down = daily_pnl[daily_pnl < 0]
    daily_sortino = daily_pnl.mean() / (down.std() + 1e-10) * np.sqrt(365) if len(down) else 0

    cum = np.array(bar_equity_curve)
    peak = np.maximum.accumulate(cum)
    dd = peak - cum
    mdd = dd.max()
    calmar = (daily_pnl.mean() * 365) / mdd if mdd > 1e-10 else 0

    status_counts = trades_df['status'].value_counts()

    mode_str = "BYPASS (Stage 1 TCN+GRU)" if bypass else "XGBoost (Stage 2)"
    print('=' * 60)
    print(f'       WALK-FORWARD BACKTEST RESULTS — {mode_str}')
    print(f'       (Leverage: {LEVERAGE}x)')
    print('=' * 60)
    print(f'  Total trades       : {n}')
    print(f'  Win rate           : {wr:.4f} ({wr*100:.1f}%)')
    print(f'  Profit factor      : {pf:.4f}')
    print(f'  Expectancy/trade   : {net.mean()*100:.4f}%')
    print(f'  Avg win            : +{avg_win*100:.4f}%')
    print(f'  Avg loss           : {avg_loss*100:.4f}%')
    print(f'  Avg bars held      : {trades_df["bars_held"].mean():.1f}')
    print()
    print(f'  Daily Sharpe       : {daily_sharpe:.4f}')
    print(f'  Daily Sortino      : {daily_sortino:.4f}')
    print(f'  Max Drawdown       : {mdd*100:.4f}%')
    print(f'  Calmar             : {calmar:.4f}')
    print()
    print(f'  Gross PnL          : {gross.sum()*100:.2f}%')
    print(f'  Net PnL            : {net.sum()*100:.2f}%')
    print(f'  Cost impact        : {(gross.sum()-net.sum())*100:.2f}%')
    print()
    print('  Close reasons:')
    for status, count in status_counts.items():
        print(f'    {status:20s}: {count:5d} ({count/n*100:.1f}%)')
    print('=' * 60)

    # Charts
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))

    cum_pct = np.cumsum(daily_pnl.values) * 100
    axes[0].plot(daily_pnl.index, cum_pct, linewidth=1, color='#2196F3')
    axes[0].fill_between(daily_pnl.index, 0, cum_pct, alpha=0.1, color='#2196F3')
    axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)
    axes[0].set_title(f'Equity Curve — {mode_str} ({LEVERAGE}x)')
    axes[0].set_ylabel('Cumulative PnL (%)')
    axes[0].grid(True, alpha=0.3)

    daily_cum = np.cumsum(daily_pnl.values)
    daily_peak = np.maximum.accumulate(daily_cum)
    daily_dd = daily_peak - daily_cum
    axes[1].fill_between(daily_pnl.index, 0, -daily_dd * 100, alpha=0.4, color='red')
    axes[1].set_title(f'Drawdown ({LEVERAGE}x)')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].grid(True, alpha=0.3)

    axes[2].hist(net * 100, bins=50, alpha=0.7, color='#4CAF50', edgecolor='white')
    axes[2].axvline(0, color='red', linestyle='--', alpha=0.7)
    axes[2].set_title('Trade PnL Distribution')
    axes[2].set_xlabel('PnL per trade (%)')
    axes[2].set_ylabel('Count')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Monthly returns
    trades_df['month'] = pd.to_datetime(trades_df['timestamp']).dt.to_period('M')
    monthly = trades_df.groupby('month')['net_pnl'].agg(['sum', 'count'])
    monthly.columns = ['return_pct', 'trades']
    monthly['return_pct'] *= 100
    print('\nMonthly Returns:')
    print(monthly.to_string())

# Dönem Bazlı Performans AnaliziWalk-forward backtest sonuçlarını **yıl ve çeyrek** bazında kırarak modelin her dönemdeaynı performansı verip vermediğini kontrol ediyoruz.

In [ ]:
if len(trades_df) > 0:
    trades_df['timestamp'] = pd.to_datetime(trades_df['timestamp'])
    trades_df['year'] = trades_df['timestamp'].dt.year
    trades_df['quarter'] = trades_df['timestamp'].dt.to_period('Q')

    print('=' * 70)
    print('       YILLIK PERFORMANS')
    print('=' * 70)

    yearly_stats = []
    for year, grp in trades_df.groupby('year'):
        net_y = grp['net_pnl'].values
        n_y = len(grp)
        wins_y = net_y[net_y > 0]
        losses_y = net_y[net_y < 0]
        wr_y = len(wins_y) / n_y if n_y > 0 else 0
        pf_y = abs(wins_y.sum() / losses_y.sum()) if len(losses_y) > 0 else float('inf')

        grp_daily = grp.groupby(grp['timestamp'].dt.date)['net_pnl'].sum()
        dr = pd.date_range(grp_daily.index.min(), grp_daily.index.max(), freq='D')
        grp_daily = grp_daily.reindex(dr, fill_value=0.0)
        sharpe_y = grp_daily.mean() / (grp_daily.std() + 1e-10) * np.sqrt(365)

        cum_y = np.cumsum(grp_daily.values)
        peak_y = np.maximum.accumulate(cum_y)
        mdd_y = (peak_y - cum_y).max()

        yearly_stats.append({
            'Yıl': year, 'İşlem': n_y, 'Win Rate': f'{wr_y*100:.1f}%',
            'Profit Factor': f'{pf_y:.2f}', 'Sharpe': f'{sharpe_y:.2f}',
            'Net PnL': f'{net_y.sum()*100:.2f}%', 'MDD': f'{mdd_y*100:.2f}%'
        })

    print(pd.DataFrame(yearly_stats).to_string(index=False))

    # Çeyreklik
    print()
    print('=' * 70)
    print('       ÇEYREKLİK PERFORMANS')
    print('=' * 70)
    quarterly_stats = []
    for q, grp in trades_df.groupby('quarter'):
        net_q = grp['net_pnl'].values
        n_q = len(grp)
        wins_q = net_q[net_q > 0]
        losses_q = net_q[net_q < 0]
        wr_q = len(wins_q) / n_q if n_q > 0 else 0
        pf_q = abs(wins_q.sum() / losses_q.sum()) if len(losses_q) > 0 else float('inf')
        quarterly_stats.append({
            'Çeyrek': str(q), 'İşlem': n_q, 'Win Rate': f'{wr_q*100:.1f}%',
            'PF': f'{pf_q:.2f}', 'Net PnL': f'{net_q.sum()*100:.2f}%'
        })
    print(pd.DataFrame(quarterly_stats).to_string(index=False))

    # Charts
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(trades_df['year'].unique())))
    for idx, (year, grp) in enumerate(trades_df.groupby('year')):
        grp_daily = grp.groupby(grp['timestamp'].dt.date)['net_pnl'].sum()
        dr = pd.date_range(grp_daily.index.min(), grp_daily.index.max(), freq='D')
        grp_daily = grp_daily.reindex(dr, fill_value=0.0)
        cum_pct = np.cumsum(grp_daily.values) * 100
        axes[0].plot(np.arange(len(cum_pct)), cum_pct, label=str(year), color=colors[idx])
    axes[0].set_title('Yıllık Kümülatif PnL')
    axes[0].set_xlabel('Gün')
    axes[0].set_ylabel('PnL (%)')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)

    years = [s['Yıl'] for s in yearly_stats]
    wrs = [float(s['Win Rate'].replace('%','')) for s in yearly_stats]
    bar_colors = ['#4CAF50' if w > 55 else '#FF9800' if w > 45 else '#F44336' for w in wrs]
    axes[1].bar(range(len(years)), wrs, color=bar_colors, edgecolor='white')
    axes[1].set_xticks(range(len(years)))
    axes[1].set_xticklabels(years)
    axes[1].set_title('Yıllık Win Rate')
    axes[1].set_ylabel('Win Rate (%)')
    axes[1].axhline(50, color='red', linestyle='--', alpha=0.5)
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print("İşlem yok — analiz yapılamadı.")